## Predição de Movimentos de Ações no Mercado Financeiro Brasileiro com LSTM e NLP: Um Estudo de Caso da PETR4 Integrando Análise de Séries Temporais e Sentimento de Notícias

# Predição de Direção de Preço da PETR4 com LSTM + NLP
> Pipeline de MLOps para Classificação de Tendência Diária de Ações

---

## 1. Fundamentação

### 1.1 Objetivo da Pesquisa
Desenvolver e avaliar um pipeline de **MLOps** para classificar a tendência diária 
de movimentação do preço da ação da Petrobras (PETR4), determinando se o ativo irá 
**subir ou cair no dia subsequente (t+1)**, por meio de duas abordagens integradas:

- **Abordagem Quantitativa:** Redes LSTM para capturar padrões temporais da série 
  histórica de preços e indicadores técnicos
- **Abordagem Qualitativa:** NLP para extração de sentimento de notícias em 
  português sobre a Petrobras e o mercado de petróleo

### 1.2 Problema de Pesquisa
> *O sentimento de notícias em português agrega valor preditivo à classificação de 
> movimentos diários da PETR4, além dos dados históricos de preço?*

### 1.3 Hipóteses
- **H0 (Nula):** O modelo com sentimento **não** apresenta desempenho 
  estatisticamente superior ao modelo baseline apenas com dados de preço
- **H1 (Alternativa):** A inclusão do score de sentimento melhora de forma 
  significativa métricas como F1-Score e Acurácia em relação ao baseline

### 1.4 Justificativa
Mercados financeiros não são puramente aleatórios reações coletivas a notícias 
geram padrões de comportamento que se refletem nos preços (Finanças Comportamentais). 
A PETR4, sendo fortemente influenciada por fatores geopolíticos, câmbio e política 
de dividendos, é particularmente sensível ao fluxo de informação em mídia.

Apesar disso, a maioria dos trabalhos acadêmicos em predição de ações utiliza 
modelos de NLP treinados em inglês. Este projeto preenche essa lacuna ao adotar 
modelos pré-treinados em **português** (BERTimbau / FinBERT-PT), tornando a análise 
de sentimento mais fidedigna ao contexto informacional real do ativo.

### 1.5 Delimitação do Escopo
| Dimensão         | Definição                                              |
|------------------|--------------------------------------------------------|
| **Ativo**        | PETR4 (Petrobras PN) — B3                              |
| **Período**      | Jan/2022 – Dez/2026                           |
| **Granularidade**| Diária (dias úteis de pregão)                          |
| **Tarefa**       | Classificação binária: Subir (1) / Cair (0)            |
| **Variável-alvo**| Sinal do retorno: `sign(Close_t+1 - Close_t)`          |
| **Cutoff NLP**   | Notícias coletadas até as 18h do dia `t` (pós-fechamento excluído) |

---

## 2. Metodologia

### 2.1 Arquitetura do Pipeline
**[Ingestão]**  ──►  yfinance, Web Scraping.    
**[Feature-Engineering]**  ──►  Indicadores técnicos, Score de sentimento, Alinhamento temporal.    
**[Modelagem]**  ──►  LSTM, GRU (ablation), XGBoost (baseline).     
**[Avaliação]**  ──►  F1, Sharpe Ratio, Backtesting.    

### 2.2 Modelos Avaliados
| Modelo       | Papel no projeto       | Justificativa                              |
|--------------|------------------------|--------------------------------------------|
| XGBoost      | Baseline               | Referência não-sequencial para comparação  |
| LSTM         | Modelo principal       | Captura dependência temporal de longo prazo|
| GRU          | Ablation study         | Variante mais leve do LSTM                 |

### 2.3 Protocolo de Validação
- **Técnica:** `TimeSeriesSplit` — sem vazamento de dados futuros
- **Splits:** 5 folds temporais (treino expande, teste avança)
- **Período de teste final (hold-out):** últimos 6 meses da série

---

## 3. Configuração do Ambiente

## Baixar PETR4

In [1]:
import yfinance as yf
import pandas as pd

ticker = "PETR4.SA"  # .SA é obrigatório para ações brasileiras (B3)
data_inicial = "2022-02-24"  # início da guerra Rússia-Ucrânia
data_final = "2026-05-26"    # hoje (crise EUA-Irã)

dados = yf.download(
    tickers=ticker,
    start=data_inicial,
    end=data_final,
    interval="1d",          # diário
    auto_adjust=True,       # ajusta dividendos e bonificações
    progress=False          # não mostrar barra de progresso
)

print(f"Baixados {len(dados)} registros de {ticker}")
print(f"Período: {dados.index.min().date()} a {dados.index.max().date()}")

print("\n Primeiras linhas:")
print(dados.head())

print("\n Colunas disponíveis:")
print(dados.columns.tolist())

Baixados 1058 registros de PETR4.SA
Período: 2022-02-24 a 2026-05-25

 Primeiras linhas:
Price           Close       High        Low       Open     Volume
Ticker       PETR4.SA   PETR4.SA   PETR4.SA   PETR4.SA   PETR4.SA
Date                                                             
2022-02-24  11.638227  12.300480  11.390753  12.129688  139674100
2022-02-25  11.850845  11.850845  11.467436  11.659141   86189100
2022-03-02  12.084377  12.300482  11.986783  12.290024   58071800
2022-03-03  11.934500  12.175002  11.906615  12.136661   69237400
2022-03-04  11.931012  12.087862  11.788105  11.878730   55418000

 Colunas disponíveis:
[('Close', 'PETR4.SA'), ('High', 'PETR4.SA'), ('Low', 'PETR4.SA'), ('Open', 'PETR4.SA'), ('Volume', 'PETR4.SA')]
